# Veritas-R1 — Stage 1 SFT (workspace curriculum, Kaggle GPU)

**Base model:** `Qwen/Qwen3-1.7B-Instruct` — same family as the production Qwen3-14B target. Native dual-mode (`enable_thinking`) chat template. Fits T4 16 GB at QLoRA r=16, batch 8, seq 512.

**Curriculum (~246K train rows across 8 datasets, see `docs/CURATED_DATASETS.md`):**
| # | Skill | Dataset | Take |
|---|---|---|---:|
| 1 | Workspace conversation (write/edit/summarise) | `HuggingFaceTB/smol-talk-2` | 60K |
| 2 | Modern instruction tuning | `allenai/tulu-3-sft-mixture` | 30K |
| 3 | Reasoning + `<think>` traces | `open-thoughts/OpenThoughts-114k` | 80K |
| 4 | Persistent memory / multi-hop | `bdsaglam/2wikimultihopqa` | 25K |
| 5 | Logical reasoning | `datatab/logiqa-2.0` + `yale-nlp/FOLIO` | 15K |
| 6 | Tool use | `NousResearch/hermes-function-calling-v1` | 11K |
| 7 | Research grounding | `bigbio/pubmed_qa` | 10K |
| 8 | Contradiction detection | `tals/vitaminc` | 15K |

**Meta-aligned regularisation stack:**
- QLoRA r=16, alpha=32, **dropout 0.10** (lower than classification — we have 250K rows)
- **NEFTune α=5** (Jain et al. 2023, used in Llama 3.1 post-training)
- **R-Drop** (Liang 2021), KL weight 0.5
- **Layer-wise LR decay** α=0.95
- Cosine LR with 5% warmup, base 2e-4
- Sample packing OFF for now (mixed-length → OK at 512 with batch 8)
- 3-seed ensemble (17, 23, 29) for downstream calibration

**Outputs:** `/kaggle/working/adapters/seed_{17,23,29}/adapter/` + `run_metrics.json`.

## 0. Environment
Internet ON. GPU T4×2 (free) or P100. ~5h per seed, 3 seeds across 2-3 sessions.

In [ ]:
%pip install --quiet --upgrade \
    "transformers>=4.46,<5" \
    "peft>=0.13" \
    "trl>=0.11" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "bitsandbytes>=0.44"

In [ ]:
import os, json, math, random, gc, sys, copy, re
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED_LIST = (17, 23, 29)
OUT_ROOT = Path("/kaggle/working")
ADAPTER_ROOT = OUT_ROOT / "adapters"
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "Qwen/Qwen3-1.7B-Instruct"
MAX_SEQ_LEN = 512
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device, "| torch =", torch.__version__)
if device == "cuda":
    print("GPU =", torch.cuda.get_device_name(0), "| mem =", torch.cuda.get_device_properties(0).total_memory // (1024**3), "GB")

## 1. Per-dataset adapters

Each adapter takes a raw HF row and returns `{"messages": [{role, content}, ...]}` in the chat-template shape. Returning `None` drops the row. The adapters are pure functions — unit-testable offline, deterministic per run.

In [ ]:
from datasets import load_dataset

SYSTEM_DEFAULT = "You are Veritas-R1, the AI inside Forge \u2014 a research workspace. Be direct, accurate, and verifiable."

# ── 1a. SmolTalk-2 (general conversation) ────────────────────
def adapter_smoltalk(row):
    msgs = row.get("messages")
    if not isinstance(msgs, list) or len(msgs) < 2: return None
    cleaned = []
    for m in msgs:
        if isinstance(m, dict) and m.get("role") in {"system","user","assistant"} and m.get("content"):
            cleaned.append({"role": m["role"], "content": str(m["content"])[:8000]})
    if not cleaned or cleaned[-1]["role"] != "assistant": return None
    if cleaned[0]["role"] != "system":
        cleaned = [{"role": "system", "content": SYSTEM_DEFAULT}] + cleaned
    return {"messages": cleaned}

# ── 1b. Tulu-3-SFT-Mixture ───────────────────────────────────
def adapter_tulu3(row):
    msgs = row.get("messages")
    if not isinstance(msgs, list) or len(msgs) < 2: return None
    cleaned = [{"role": m["role"], "content": str(m.get("content",""))[:8000]} for m in msgs if isinstance(m, dict) and m.get("role") in {"system","user","assistant"} and m.get("content")]
    if not cleaned or cleaned[-1]["role"] != "assistant": return None
    if cleaned[0]["role"] != "system":
        cleaned = [{"role": "system", "content": SYSTEM_DEFAULT}] + cleaned
    return {"messages": cleaned}

# ── 1c. OpenThoughts-114k (reasoning + <think>) ─────────────
_BEGIN_THOUGHT, _END_THOUGHT = "<|begin_of_thought|>", "<|end_of_thought|>"
_BEGIN_SOLUTION, _END_SOLUTION = "<|begin_of_solution|>", "<|end_of_solution|>"
def _split_openthoughts(text):
    thought = text.split(_BEGIN_THOUGHT,1)[1].split(_END_THOUGHT,1)[0].strip() if _BEGIN_THOUGHT in text and _END_THOUGHT in text else None
    if _BEGIN_SOLUTION in text and _END_SOLUTION in text:
        answer = text.split(_BEGIN_SOLUTION,1)[1].split(_END_SOLUTION,1)[0].strip()
    elif _END_THOUGHT in text:
        answer = text.split(_END_THOUGHT,1)[1].strip()
    else:
        answer = text.strip()
    return thought, answer

def adapter_openthoughts(row):
    conv = row.get("conversations") or []
    if not isinstance(conv, list): return None
    user_text, asst_text = None, None
    for t in conv:
        if not isinstance(t, dict): continue
        if t.get("from") in {"user","human"} and not user_text:
            user_text = t.get("value")
        elif t.get("from") in {"assistant","gpt"} and user_text:
            asst_text = t.get("value"); break
    if not user_text or not asst_text: return None
    thought, answer = _split_openthoughts(asst_text)
    if not answer: return None
    # Combine into a `<think>...</think>` reasoning block + final answer.
    if thought:
        content = f"<think>\n{thought}\n</think>\n\n{answer}"
    else:
        content = answer
    return {"messages": [
        {"role": "system", "content": SYSTEM_DEFAULT},
        {"role": "user", "content": str(user_text)[:6000]},
        {"role": "assistant", "content": content[:8000]},
    ]}

# ── 1d. 2WikiMultiHopQA (memory / multi-hop) ────────────────
def adapter_wikihop(row):
    q = row.get("question"); a = row.get("answer")
    ctx = row.get("context") or row.get("contexts") or ""
    if not q or not a: return None
    if isinstance(ctx, list):
        ctx = "\n\n".join(str(c) for c in ctx[:5])
    prompt = f"Use the supporting passages to answer.\n\nPassages:\n{str(ctx)[:4500]}\n\nQuestion: {q}"
    return {"messages": [
        {"role": "system", "content": SYSTEM_DEFAULT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": str(a)[:1500]},
    ]}

# ── 1e. LogiQA 2.0 (logical reasoning) ──────────────────────
def adapter_logiqa(row):
    passage = row.get("context") or row.get("passage") or ""
    q = row.get("question") or row.get("query")
    options = row.get("options") or row.get("choices")
    answer_idx = row.get("answer") if isinstance(row.get("answer"), int) else row.get("label")
    if not q or not options or answer_idx is None: return None
    if not isinstance(options, list): return None
    letters = ["A","B","C","D","E"][:len(options)]
    formatted_opts = "\n".join(f"{l}) {o}" for l, o in zip(letters, options))
    user = f"Read the passage, apply the relevant logic, and pick the best answer.\n\n{passage}\n\nQuestion: {q}\n\n{formatted_opts}"
    asst = f"{letters[answer_idx]}"
    return {"messages": [
        {"role": "system", "content": SYSTEM_DEFAULT},
        {"role": "user", "content": user[:6000]},
        {"role": "assistant", "content": asst},
    ]}

# ── 1f. Hermes Function Calling v1 ──────────────────────────
def adapter_hermes_fc(row):
    conv = row.get("conversations") or []
    if not isinstance(conv, list) or len(conv) < 2: return None
    msgs = []
    for t in conv:
        if not isinstance(t, dict): continue
        f = t.get("from"); v = t.get("value")
        if not v: continue
        role = {"system":"system","human":"user","user":"user","gpt":"assistant","assistant":"assistant","tool":"tool"}.get(f)
        if role and role in {"system","user","assistant","tool"}:
            msgs.append({"role": role, "content": str(v)[:6000]})
    if not msgs or msgs[-1]["role"] != "assistant": return None
    if msgs[0]["role"] != "system":
        msgs = [{"role": "system", "content": SYSTEM_DEFAULT}] + msgs
    return {"messages": msgs}

# ── 1g. PubMedQA labeled ─────────────────────────────────────
def adapter_pubmedqa(row):
    q = row.get("question") or row.get("QUESTION")
    long_answer = row.get("long_answer") or row.get("LONG_ANSWER")
    contexts = row.get("contexts") or row.get("CONTEXTS")
    if not q or not long_answer or not contexts: return None
    if not isinstance(contexts, list): return None
    ctx_str = "\n\n".join(str(c) for c in contexts)[:5000]
    user = f"Use the source passages to answer.\n\nSources:\n{ctx_str}\n\nQuestion: {q}"
    return {"messages": [
        {"role": "system", "content": SYSTEM_DEFAULT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": str(long_answer)[:3000]},
    ]}

# ── 1h. Vitamin-C (contradiction) ───────────────────────────
def adapter_vitaminc(row):
    claim, evidence, label = row.get("claim"), row.get("evidence"), row.get("label")
    if not all(isinstance(x, str) and x for x in (claim, evidence, label)): return None
    label_n = label.strip().upper()
    if label_n == "SUPPORTS":         verdict = "SUPPORT"; reason = "The evidence directly supports the claim."
    elif label_n == "REFUTES":        verdict = "CONTRADICT"; reason = "The evidence contradicts the claim."
    elif label_n == "NOT ENOUGH INFO": verdict = "NOT_ENOUGH_INFO"; reason = "The evidence is insufficient."
    else: return None
    user = f"Claim: {claim}\n\nEvidence: {evidence}\n\nDoes the evidence SUPPORT, CONTRADICT, or fail to settle the claim? Respond with one verdict word and one short rationale."
    return {"messages": [
        {"role": "system", "content": SYSTEM_DEFAULT},
        {"role": "user", "content": user[:5000]},
        {"role": "assistant", "content": f"{verdict}: {reason}"},
    ]}

## 2. Pull the curriculum and assemble the train/val mixes

Streaming `load_dataset` so we don't blow up disk. Each source caps at `target_count` valid rows, then we concatenate, shuffle with seed, and split into train/val/test.

In [ ]:
REGISTRY = [
    {"name": "smoltalk2",       "hf": "HuggingFaceTB/smol-talk-2",                "split": "train", "config": None,                                  "adapter": adapter_smoltalk,    "take": 60_000, "val": 1_000, "test": 1_000},
    {"name": "tulu3",           "hf": "allenai/tulu-3-sft-mixture",                "split": "train", "config": None,                                  "adapter": adapter_tulu3,       "take": 30_000, "val":   500, "test":   500},
    {"name": "openthoughts",    "hf": "open-thoughts/OpenThoughts-114k",           "split": "train", "config": None,                                  "adapter": adapter_openthoughts,"take": 80_000, "val": 1_500, "test": 1_500},
    {"name": "wikihop",         "hf": "bdsaglam/2wikimultihopqa",                  "split": "train", "config": None,                                  "adapter": adapter_wikihop,     "take": 25_000, "val":   300, "test":   300},
    {"name": "logiqa2",         "hf": "datatab/logiqa-2.0",                        "split": "train", "config": None,                                  "adapter": adapter_logiqa,      "take": 15_000, "val":   500, "test":   500},
    {"name": "hermes_fc",       "hf": "NousResearch/hermes-function-calling-v1",   "split": "train", "config": None,                                  "adapter": adapter_hermes_fc,   "take": 11_000, "val":   100, "test":   100},
    {"name": "pubmedqa",        "hf": "bigbio/pubmed_qa",                          "split": "train", "config": "pubmed_qa_labeled_fold0_source",      "adapter": adapter_pubmedqa,    "take": 10_000, "val":   200, "test":   200},
    {"name": "vitaminc",        "hf": "tals/vitaminc",                              "split": "train", "config": None,                                  "adapter": adapter_vitaminc,    "take": 15_000, "val":   500, "test":   500},
]

def pull_source(spec, seed):
    print(f"-- {spec['name']:<14}  hf={spec['hf']}  take={spec['take']}")
    kwargs = {"streaming": True}
    if spec["config"]: kwargs["name"] = spec["config"]
    ds = load_dataset(spec["hf"], split=spec["split"], **kwargs)
    rng = random.Random(seed + abs(hash(spec["name"])) % 99991)
    out = []
    needed = spec["take"] + spec["val"] + spec["test"]
    for row in ds:
        ex = spec["adapter"](dict(row))
        if ex is not None:
            ex["_source"] = spec["name"]
            out.append(ex)
            if len(out) >= needed * 1.2:  # 20% slack for randomization
                break
    rng.shuffle(out)
    train = out[: spec["take"]]
    val = out[spec["take"]: spec["take"] + spec["val"]]
    test = out[spec["take"] + spec["val"]: spec["take"] + spec["val"] + spec["test"]]
    print(f"   collected={len(out)} train={len(train)} val={len(val)} test={len(test)}")
    return train, val, test

train_pool, val_pool, test_pool = [], [], []
for spec in REGISTRY:
    tr, va, te = pull_source(spec, seed=17)
    train_pool.extend(tr); val_pool.extend(va); test_pool.extend(te)

rng_master = random.Random(17)
rng_master.shuffle(train_pool); rng_master.shuffle(val_pool); rng_master.shuffle(test_pool)
print(f"\nTOTAL  train={len(train_pool)}  val={len(val_pool)}  test={len(test_pool)}")

## 3. Tokenise with chat template + assistant-only loss mask

We mask EVERYTHING except the assistant turn so loss only fires on what we want the model to learn. This is the standard SFT recipe used in Llama 3.1, Tulu 3, and OpenHermes.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenise_one(example, max_len=MAX_SEQ_LEN):
    full = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    prefix_msgs = example["messages"][:-1]
    prefix = tokenizer.apply_chat_template(prefix_msgs, tokenize=False, add_generation_prompt=True)
    full_ids = tokenizer(full, truncation=True, max_length=max_len, add_special_tokens=False)["input_ids"]
    pre_ids = tokenizer(prefix, truncation=True, max_length=max_len, add_special_tokens=False)["input_ids"]
    labels = list(full_ids)
    for i in range(min(len(pre_ids), len(labels))):
        labels[i] = -100
    if sum(1 for x in labels if x != -100) < 4: return None
    return {"input_ids": full_ids, "attention_mask": [1]*len(full_ids), "labels": labels}

def tokenise_pool(pool, max_len=MAX_SEQ_LEN, name=""):
    out = []; drops = 0
    for ex in pool:
        t = tokenise_one(ex, max_len=max_len)
        if t is None: drops += 1; continue
        out.append(t)
    print(f"  tokenised {name}: {len(out)} (drops={drops})")
    return out

tok_train = tokenise_pool(train_pool, name="train")
tok_val   = tokenise_pool(val_pool, name="val")

## 4. Model + LoRA + R-Drop loss + LLRD

Per Meta's Llama 3.1 §3, we add **NEFTune α=5** (random embed noise during training) on top of the LoRA adapter — ~+1-2 points on AlpacaEval2 / MT-Bench for free.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

def build_model(seed):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2" if device == "cuda" else "eager",
    )
    model = prepare_model_for_kbit_training(model)
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16, lora_alpha=32, lora_dropout=0.10, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    
    # NEFTune — random embed-noise during training only
    embed = model.get_input_embeddings()
    NEFTUNE_ALPHA = 5.0
    def hook(_m, _inp, output):
        if not model.training: return output
        seq_len, dim = output.shape[-2], output.shape[-1]
        scale = NEFTUNE_ALPHA / (seq_len * dim) ** 0.5
        noise = torch.zeros_like(output).uniform_(-scale, scale)
        return output + noise
    embed.register_forward_hook(hook)
    return model.to(device)

def rdrop_loss(logits_a, logits_b, labels, kl_weight=0.5):
    # Causal-LM cross-entropy: shift one position, ignore -100 mask.
    sa = logits_a[..., :-1, :].contiguous(); sb = logits_b[..., :-1, :].contiguous()
    lab = labels[..., 1:].contiguous()
    ce_a = F.cross_entropy(sa.view(-1, sa.size(-1)), lab.view(-1), ignore_index=-100)
    ce_b = F.cross_entropy(sb.view(-1, sb.size(-1)), lab.view(-1), ignore_index=-100)
    log_pa = F.log_softmax(sa, dim=-1); log_pb = F.log_softmax(sb, dim=-1)
    pa = log_pa.exp(); pb = log_pb.exp()
    kl = 0.5 * (F.kl_div(log_pb, pa, reduction="batchmean") + F.kl_div(log_pa, pb, reduction="batchmean"))
    return 0.5*(ce_a+ce_b) + kl_weight * kl

def build_llrd_groups(model, base_lr=2e-4, decay=0.95, weight_decay=0.0):
    layer_re = re.compile(r"\.layers\.(\d+)\.")
    n_layers = max((int(m.group(1)) for n, _ in model.named_parameters() for m in [layer_re.search(n)] if m), default=0) + 1
    groups = []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        m = layer_re.search(name)
        if m:
            depth = max(0, n_layers - 1 - int(m.group(1)))
            lr = base_lr * (decay ** depth)
        else:
            lr = base_lr
        wd = 0.0 if any(k in name for k in ("bias","LayerNorm","layernorm","norm")) else weight_decay
        groups.append({"params":[p], "lr": lr, "weight_decay": wd})
    print(f"LLRD: {n_layers} layers | {len(groups)} groups | base_lr={base_lr}")
    return groups

## 5. Padding collator + DataLoader

In [ ]:
from torch.utils.data import DataLoader, Dataset

class TokenisedDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    max_len = max(len(b["input_ids"]) for b in batch)
    pad_id = tokenizer.pad_token_id
    iids, attn, lab = [], [], []
    for b in batch:
        pad = max_len - len(b["input_ids"])
        iids.append(b["input_ids"] + [pad_id]*pad)
        attn.append(b["attention_mask"] + [0]*pad)
        lab.append(b["labels"] + [-100]*pad)
    return {
        "input_ids":      torch.tensor(iids, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "labels":         torch.tensor(lab, dtype=torch.long),
    }

def make_loader(items, batch_size, shuffle, seed=0):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(TokenisedDataset(items), batch_size=batch_size, shuffle=shuffle, generator=g, drop_last=False, num_workers=2, pin_memory=(device=="cuda"), collate_fn=collate)

## 6. Training loop with early stopping

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    losses, n_total = 0.0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        losses += float(out.loss) * batch["input_ids"].size(0)
        n_total += batch["input_ids"].size(0)
    return {"nll": losses / max(n_total,1)}

def train_one_seed(seed, epochs=2, batch_size=8, base_lr=2e-4, eval_every=500, patience=2):
    print(f"\n=== seed {seed} ===")
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model = build_model(seed)
    groups = build_llrd_groups(model, base_lr=base_lr, decay=0.95, weight_decay=0.0)
    optim = torch.optim.AdamW(groups, lr=base_lr)
    train_loader = make_loader(tok_train, batch_size, shuffle=True, seed=seed)
    val_loader   = make_loader(tok_val, batch_size*2, shuffle=False)
    total_steps = len(train_loader) * epochs
    warmup = int(0.05 * total_steps)
    def lr_factor(s):
        if s < warmup: return s / max(1, warmup)
        prog = (s - warmup) / max(1, total_steps - warmup)
        return 0.5 * (1 + math.cos(math.pi * prog))
    sched = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda=lr_factor)
    
    best, misses, step, hist = float("inf"), 0, 0, []
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            step += 1
            batch = {k: v.to(device) for k, v in batch.items()}
            out_a = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            out_b = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            loss = rdrop_loss(out_a.logits, out_b.logits, batch["labels"], kl_weight=0.5)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
            optim.step(); sched.step(); optim.zero_grad()
            if step % 100 == 0:
                print(f"step {step:>5}/{total_steps} | loss {float(loss):.4f} | lr {sched.get_last_lr()[0]:.2e}")
            if step % eval_every == 0 or step == total_steps:
                v = evaluate(model, val_loader)
                hist.append({"step": step, "val_nll": v["nll"]})
                print(f"  -> val_nll {v['nll']:.4f}")
                if v["nll"] < best - 1e-3:
                    best = v["nll"]; misses = 0
                else:
                    misses += 1
                    if misses >= patience:
                        print(f"  -> early-stop at step {step}"); break
                model.train()
        if misses >= patience: break
    out_dir = ADAPTER_ROOT / f"seed_{seed}" / "adapter"
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    summary = {"seed": seed, "best_val_nll": float(best), "history": hist, "steps": step}
    (out_dir.parent / "summary.json").write_text(json.dumps(summary, indent=2))
    print(f"[seed {seed}] best_val_nll={best:.4f} saved to {out_dir}")
    del model; gc.collect(); torch.cuda.empty_cache()
    return summary

## 7. Run the 3-seed ensemble (across multiple Kaggle sessions if needed)

If you only have time for one seed in a single session, just run `train_one_seed(17)` and come back later for 23 + 29. The output structure is identical and the eval notebook glues them together.

In [ ]:
ensemble = []
for s in SEED_LIST:
    if (ADAPTER_ROOT / f"seed_{s}" / "summary.json").is_file():
        print(f"seed {s} already trained — skipping")
        ensemble.append(json.loads((ADAPTER_ROOT / f"seed_{s}" / "summary.json").read_text()))
        continue
    ensemble.append(train_one_seed(s))
(OUT_ROOT / "run_metrics.json").write_text(json.dumps({"seeds": SEED_LIST, "members": ensemble}, indent=2))
print("\nAll seeds done.")

## 8. Next steps
1. Run `veritas-stage1-augment.ipynb` (optional, +25% synthetic paraphrase pool) to grow the dataset using the SFT'd model itself.
2. Run `veritas-stage3-simpo.ipynb` for 3 rounds of iterative SimPO on the strongest seed (or the prob-averaged ensemble).
3. Final eval/calibration via existing `veritas-stage1-eval.ipynb` (works on the saved logits).